# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [7]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [8]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf")
docs = loader.load()

document_text = ""
for page in docs:
    # .strip() removes leading/trailing whitespace from each page
    document_text += page.page_content.strip() + "\n\n"

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [9]:
import os
from openai import OpenAI
from pydantic import BaseModel, Field

# Initialize the client with the API Gateway URL
client = OpenAI(
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    api_key="gateway-auth" 
)

class SummaryEvaluation(BaseModel):
    Author: str
    Title: str
    Relevance: str = Field(description="Relevance to an AI professional (max 1 paragraph).")
    Summary: str = Field(description="The summary of the article (max 1000 tokens).")
    Tone: str = Field(description="The specific tone used for the summary.")
    InputTokens: int = Field(default=0)
    OutputTokens: int = Field(default=0)
    
# Separate instructions and context
developer_instructions = "You are a professional editor. Extract information and summarize the provided text."
tone_style = "Bureaucratese"

#DEFINE PROMPT TEMPLATES
USER_CONTEXT_TEMPLATE = """
    "Please analyze the provided text and generate a summary."
    "The summary MUST be written in the style of {tone_style}."
    "The JSON must include exactly these keys: author, title, relevance, summary, tone."

TEXT TO PROCESS:
{text_content}
"""
user_content = USER_CONTEXT_TEMPLATE.format(
    tone_style=tone_style,
    text_content=document_text)
completion = client.beta.chat.completions.parse(
    model="gpt-4o", 
    messages=[
        # Developer role provides instructions/behavior
        {"role": "developer", "content": developer_instructions},
        # User role provides the dynamic context/data
        {"role": "user", "content": user_content},
    ],
    response_format=SummaryEvaluation,
)

summary_object = completion.choices[0].message.parsed

summary_object.InputTokens = completion.usage.prompt_tokens
summary_object.OutputTokens = completion.usage.completion_tokens

print(summary_object)

Author='Peter F. Drucker' Title='Managing Oneself' Relevance='For an AI professional, understanding how to manage oneself—particularly in a long-term, knowledge-based career—is crucial. With careers increasingly characterized by flexibility and self-management, the article provides valuable insights into aligning personal strengths, values, and work styles to achieve sustained excellence and satisfaction.' Summary="Pursuant to the esteemed author Peter F. Drucker's exposition in the Harvard Business Review edition, referred to colloquially as 'Managing Oneself,' the article delineates the paramount significance of self-understanding and development in the contemporary knowledge economy. Notably, individuals must assume the role of Chief Executive Officers over their careers given the diminishing involvement of traditional corporate structures in career management. The onus, therefore, lies on individuals to identify and capitalize on their intrinsic strengths while acknowledging and mi

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [10]:
class EvaluationResult(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str


# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [11]:
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.models import GPTModel

# Initialize all required metrics
custom_eval_model = GPTModel(
    model="gpt-4o",
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    _openai_api_key="gateway-auth", 
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')}
)

summ_metric = SummarizationMetric(threshold=0.5, model=custom_eval_model, assessment_questions=[
    "Does the summary identify the author Peter Drucker?",
    "Does the summary mention 'Managing Oneself'?",
    "Is the summary concise and under 1000 tokens?",
    "Does it explain relevance for AI professionals?",
    "Are there any factual contradictions?"
])

# Implementing the three G-Eval metrics as requested
coherence_metric = GEval(name="Coherence", model=custom_eval_model, criteria="Coherence and clarity of the summary.", evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT])
tonality_metric = GEval(name="Tonality", model=custom_eval_model, criteria=f"Adherence to {tone_style} style.", evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT])
safety_metric = GEval(name="Safety", model=custom_eval_model, criteria="Ensure no harmful or biased content.", evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT])

# Initial Evaluation
test_case = LLMTestCase(input=document_text, actual_output=summary_object.Summary, retrieval_context=[document_text])

for metric in [summ_metric, coherence_metric, tonality_metric, safety_metric]:
    metric.measure(test_case)

# Report Initial Results
initial_results = EvaluationResult(
    SummarizationScore=summ_metric.score, SummarizationReason=summ_metric.reason,
    CoherenceScore=coherence_metric.score, CoherenceReason=coherence_metric.reason,
    TonalityScore=tonality_metric.score, TonalityReason=tonality_metric.reason,
    SafetyScore=safety_metric.score, SafetyReason=safety_metric.reason
)
print("--- INITIAL EVALUATION ---")
print(initial_results.model_dump_json(indent=4))

# Enhancement Step
class EnhancedSummary(SummaryEvaluation):
    Reasoning: str = Field(description="Detailed explanation of how the summary was corrected.")

enhancement_prompt = f"Refine this summary. Critique: {summ_metric.reason}. Style: {tone_style}. Text: {document_text[:2000]}"
enhancement_comp = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": enhancement_prompt}],
    response_format=EnhancedSummary
)
enhanced_obj = enhancement_comp.choices[0].message.parsed

#Final Evaluation of Enhanced Summary
enhanced_test_case = LLMTestCase(input=document_text, actual_output=enhanced_obj.Summary, retrieval_context=[document_text])

for metric in [summ_metric, coherence_metric, tonality_metric, safety_metric]:
    metric.measure(enhanced_test_case)

enhanced_results = EvaluationResult(
    SummarizationScore=summ_metric.score, SummarizationReason=summ_metric.reason,
    CoherenceScore=coherence_metric.score, CoherenceReason=coherence_metric.reason,
    TonalityScore=tonality_metric.score, TonalityReason=tonality_metric.reason,
    SafetyScore=safety_metric.score, SafetyReason=safety_metric.reason
)
print("\n--- ENHANCED EVALUATION ---")
print(enhanced_results.model_dump_json(indent=4))

Output()

Output()

Output()

Output()

--- INITIAL EVALUATION ---
{
    "SummarizationScore": 0.6428571428571429,
    "SummarizationReason": "The score is 0.64 because the summary contains a contradiction by suggesting a focus on addressing weaknesses, which is not aligned with the original text's emphasis on improving strengths. Additionally, the summary introduces several pieces of extra information, such as feedback analysis, strategic contributions, and career diversification, which are not present in the original text. These discrepancies and additions reduce the alignment and accuracy of the summary with the original content.",
    "CoherenceScore": 0.897404264374409,
    "CoherenceReason": "The response demonstrates a coherent logical flow, clearly outlining Drucker's ideas on self-management in the knowledge economy. The language is clear and precise, making complex concepts easily understandable. Terminology is consistent, with repeated emphasis on self-understanding, career management, and personal strengths. Ther

Output()

Output()

Output()

Output()


--- ENHANCED EVALUATION ---
{
    "SummarizationScore": 0.0,
    "SummarizationReason": "The score is 0.00 because the summary includes extra information not present in the original text, such as ambition and intelligence as factors for career success, a 50-year work span, and the need for ongoing engagement and productivity. Additionally, the summary fails to identify the author Peter Drucker and does not mention 'Managing Oneself', which are details available in the original text.",
    "CoherenceScore": 0.8822189126165023,
    "CoherenceReason": "The response demonstrates a coherent logical flow, starting with the importance of ambition and intelligence, and transitioning to the necessity of self-leadership in career management. The language is clear and easily understandable, with consistent terminology related to career development. There are no ambiguities or contradictions present, making the output well-aligned with the evaluation steps.",
    "TonalityScore": 0.76525448612047

#### Report your results. Did you get a better output? 

+ Evaluate the new summary using the same function.
This is the results of this evaluation:
    - SUMMARIZATION: 0.64 -> 0.0
    - COHERENCE:     0.89 -> 0.88
    - TONALITY:      0.90 -> 0.76
    - SAFETY:        0.98 -> 0.95 
+ Summarization: Decreased in the enhanced version: Went from 0.64 to 0
+ Coherence and Safety: Values stayed relatively the same
+ Tonality decreased: This could have decreased as the model was prioritizing improving other feedback over maintaining the tone to ensure that the other parameters such as coherence and summarization performed better

Other reasons why this could have happened:
+ Over-Correction: The model tried to fix the "extra information" that was flagged for feedback
+ New Hallucinations: Despite the critique, the model introduced external concepts in the original source.

Why? Do you think these controls are enough?
+ These controls may not be enough for specific use cases of LLMs. If all of these parameters of summarization, coherence and tonality were equally important, the model may not have the ability to perform as desired. Additionally prompting may be required.

Resources: https://arxiv.org/pdf/2203.02155


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
